# Student Performance Regression — Python walkthrough

**Aazim Ashraf · Data Science · Python and R**

Learn to predict a final grade using earlier grades. We compare a constant baseline, simple regression and multiple regression. The original [UCI mathematics dataset](https://archive.ics.uci.edu/dataset/320/student+performance) and a fixed train/test split are bundled with this repository.

Open this notebook from the project root or the `notebooks` folder. Install the project requirements in your notebook environment first. All eight code cells below have been executed; the saved output is an example run.


## 1. Load the project

The reusable functions are in `python/regression.py`. Importing them avoids maintaining two different versions of the analysis.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data/student-mat.csv").exists():
    ROOT = ROOT.parent
if not (ROOT / "data/student-mat.csv").exists():
    raise FileNotFoundError("Open this notebook inside the project folder.")
sys.path.insert(0, str(ROOT / "python"))
from regression import load_data, fit_models, score_models, coefficient_table, run_analysis
from predict import predict_grade
print("Project imports ready.")


Project imports ready.


## 2. Validate the data and separate train from test

G1 and G2 are earlier grades; G3 is the target final grade. All use a 0–20 scale. The split uses original row numbers rather than a new random split in each language.


In [2]:
students = load_data(ROOT)
train = students.loc[students["split"] == "train"].copy()
test = students.loc[students["split"] == "test"].copy()
print(f"Training records: {len(train)}; test records: {len(test)}")
print(f"Zero final grades retained: {(students['G3'] == 0).sum()}")
print(train[["row_id", "G1", "G2", "G3"]].head().to_string(index=False))
assert not set(train.row_id) & set(test.row_id)


Training records: 316; test records: 79
Zero final grades retained: 38
 row_id  G1  G2  G3
      2   5   5   6
      4  15  14  15
      5   6  10  10
      6  15  15  15
      7  12  12  11


## 3. Inspect the training records

Explore training records before fitting. Repeatedly using test outcomes to choose features or settings would weaken their role as an independent check.


In [3]:
print("Training grade summaries:")
print(train[["G1", "G2", "G3"]].describe().round(2).to_string())
print("\nTraining correlations:")
print(train[["G1", "G2", "G3"]].corr().round(3).to_string())


Training grade summaries:
           G1      G2      G3
count  316.00  316.00  316.00
mean    10.94   10.72   10.47
std      3.38    3.88    4.60
min      3.00    0.00    0.00
25%      8.00    9.00    8.00
50%     11.00   11.00   11.00
75%     13.00   13.00   14.00
max     19.00   19.00   20.00

Training correlations:
       G1     G2     G3
G1  1.000  0.850  0.802
G2  0.850  1.000  0.915
G3  0.802  0.915  1.000


## 4. Fit the models

The baseline predicts the training-target mean. Simple regression uses `G2`. Multiple regression uses `G1` and `G2`. G3 is never supplied as an input.

`predicted G3 = intercept + coefficient_1 × G1 + coefficient_2 × G2`


In [4]:
models = fit_models(train)
print(coefficient_table(models).to_string(index=False, float_format=lambda value: f"{value:.5f}"))


         model      term  estimate
 mean_baseline intercept  10.46835
     simple_G2 intercept  -1.15504
     simple_G2        G2   1.08476
multiple_G1_G2 intercept  -1.51114
multiple_G1_G2        G1   0.11880
multiple_G1_G2        G2   0.99673


## 5. Evaluate the same held-out records

MAE is the average absolute error; RMSE responds more strongly to large errors. Both are grade points. R² compares squared errors against deviations from the test-target mean; it is not an accuracy percentage. No predictions are rounded or clipped for scoring.


In [5]:
metrics, predictions = score_models(models, test)
print(metrics.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("\nExamples from the multiple model:")
print(predictions[predictions.model == "multiple_G1_G2"].head().round(3).to_string(index=False))


         model  test_rows    mae   rmse      r2
 mean_baseline         79 3.3644 4.5177 -0.0035
     simple_G2         79 1.3624 2.3076  0.7382
multiple_G1_G2         79 1.3448 2.2721  0.7462

Examples from the multiple model:
 row_id          model  actual  predicted  residual
      1 multiple_G1_G2       6      5.063     0.937
      3 multiple_G1_G2      10      7.294     2.706
      9 multiple_G1_G2      19     18.331     0.669
     27 multiple_G1_G2      11     11.875    -0.875
     45 multiple_G1_G2       9      9.644    -0.644


## 6. Verify the linear algebra

The design matrix has a column of ones for the intercept, followed by G1 and G2. `np.linalg.lstsq` solves the least-squares problem directly. It avoids explicitly calculating a matrix inverse.


In [6]:
X_train = np.column_stack([np.ones(len(train)), train[["G1", "G2"]].to_numpy()])
X_test = np.column_stack([np.ones(len(test)), test[["G1", "G2"]].to_numpy()])
beta, *_ = np.linalg.lstsq(X_train, train["G3"].to_numpy(), rcond=None)
manual_predictions = X_test @ beta
sklearn_predictions = models["multiple_G1_G2"].predict(test[["G1", "G2"]])
np.testing.assert_allclose(manual_predictions, sklearn_predictions, rtol=1e-10, atol=1e-10)
print("Coefficients [intercept, G1, G2]:", np.round(beta, 6))
print("Independent least squares and scikit-learn predictions match.")


Coefficients [intercept, G1, G2]: [-1.511135  0.118801  0.996734]
Independent least squares and scikit-learn predictions match.


## 7. Save the outputs and inspect the plots

This command recreates the CSV results, the saved equation, and the overview figure. It uses the exact same training and test split as the steps above.


In [7]:
saved_metrics = run_analysis(ROOT)


         model  test_rows    mae   rmse      r2
 mean_baseline         79 3.3644 4.5177 -0.0035
     simple_G2         79 1.3624 2.3076  0.7382
multiple_G1_G2         79 1.3448 2.2721  0.7462

Outputs saved in reports/python and reports/figures.


![Regression overview](../reports/figures/regression_overview.svg)

The upper-left panel uses training records. The other panels use the 79 test records. Large negative residuals mean the model overpredicted a final grade. Zero final grades remain in the plot and in the metrics.


## 8. Try a hypothetical example

This is an illustration, not a known student's result. Earlier grades must already be available. Individual outcomes remain uncertain.


In [8]:
estimated_grade = predict_grade(12, 14, ROOT / "reports/python/model.json")
print(f"For G1 = 12 and G2 = 14, predicted G3 = {estimated_grade:.2f} / 20.")


For G1 = 12 and G2 = 14, predicted G3 = 13.87 / 20.


## Run the equivalent R analysis

In RStudio, open the repository's `.Rproj` file and run:

```r
source("R/analysis.R")
```

After running both versions, compare their outputs from a terminal at the repository root:

```bash
python scripts/compare_languages.py
```

## What the result means

The multiple model has a test MAE of about 1.34 grade points and an R² of about 0.746. Its improvement over G2 alone is small on this one split. This is a late-term model using a small historical school dataset; it has not been evaluated on IUST students. Correlations and coefficients do not establish causal effects.

Source credit: Cortez, P. (2008). *Student Performance* [Dataset]. UCI Machine Learning Repository. [doi:10.24432/C5TG7T](https://doi.org/10.24432/C5TG7T). Dataset license: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
